# NanoGPT Slowrun: current tiny-track recipe, Colab study notebook

**Checked October 19, 2026.** Latest accepted tiny entry: **3.295 validation
loss in 14.2 minutes on 8xH100**, submitted September 5, 2026. This notebook
has not achieved that result. [Leaderboard](https://github.com/qlabs-eng/slowrun#tiny-track-15-minutes)
| [Record PR](https://github.com/qlabs-eng/slowrun/pull/99)
| [Pinned upstream code](https://github.com/qlabs-eng/slowrun/tree/c6f0fa1b415994e0c5cf163990fe253fe1622883)

Two paths are provided:

- **Portable study path:** self-contained PyTorch/SDPA code, scaled presets,
  current algorithmic features, model summary, memory/time probe, clean
  validation, EMA/SWA, checkpoints, generation and an optional MuonEq-R A/B test.
- **Official recipe path:** a guarded launcher for the unmodified pinned
  upstream trainer and canonical data on an 8xH100 host. Use this path for
  actual record reproduction. It retains the FP8 CUDA loss and distributed recipe.

Choose **Runtime > Change runtime type > GPU**. Start with `smoke`, then use
the probe before increasing the model. T4 uses FP32 by default because it has
no native BF16; A100/L4 use BF16 when supported. No package cell replaces Colab's
PyTorch installation. A reduced-data or single-GPU result is not an official tiny score.

Your original exported `.py` is preserved. Stale execution outputs have not
been copied into this notebook. The definitions cell is visible/editable;
the full methods guide appears at the end.


## What changed from the older Colab port?

| Feature | Current notebook behavior |
|---|---|
| Learnable exclusive self-attention | Per-head subtraction of the value-aligned output component; all layers by default. |
| Recurrence | One network pass, then two late in training; backprop through the last pass. |
| MTP | **One shared head**, weighted shifted targets, with extra offsets annealed away. No additional vocabulary heads. |
| Meta-gradient | Temporary normalized MLP adaptation; query gradients taken at adapted weights; exact restoration; supports accumulated study batches. |
| Schedules | Current warmdown, hold/decay/ramp weight decay, Muon momentum warmup, late cosine SWA. |
| Averaging | CPU EMA for study VRAM savings, bias correction, and recency-weighted late checkpoint averaging. |
| Data and evaluation | Fixed canonical validation prefix, metadata/hash-aware cache, deterministic clean CE/BPB, exact processed-token accounting. |
| Extra experiment | MuonEq-R row balancing, **off by default**. |

**Numerical differences:** ordinary autograd instead of fused FP8 CE, SDPA
instead of FA3, FP32 master parameters, single-device optimizer and EMA storage.
Short final accumulation groups are processed exactly. These prevent treating
the portable implementation as a bitwise or runtime reproduction of the leader.


In [ ]:
#@title 1. Dependencies and runtime
import importlib.util
import subprocess
import sys

required = {"numpy": "numpy", "tiktoken": "tiktoken", "datasets": "datasets", "tqdm": "tqdm", "matplotlib": "matplotlib"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
import torch
print("PyTorch:", torch.__version__)
print("CUDA devices:", torch.cuda.device_count())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(props.name, f"{props.total_memory / 2**30:.2f} GiB")
else:
    print("CPU mode is available for synthetic checks; select a Colab GPU for real experiments.")


In [ ]:
#@title 2. Current tiny recipe: model, optimizer, data and training definitions
"""Single-device study port of Q Labs Slowrun tiny, pinned 2026-10-19.

Upstream: https://github.com/qlabs-eng/slowrun/tree/c6f0fa1b415994e0c5cf163990fe253fe1622883
This portable SDPA/ordinary-precision implementation is NOT the H100 record runner.

MIT License
Copyright (c) 2025 [Your Name or Organization]
Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:
The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.
THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""

from __future__ import annotations

import csv
import gc
import hashlib
import json
import math
import random
import time
from contextlib import contextmanager, nullcontext
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

UPSTREAM_COMMIT = "c6f0fa1b415994e0c5cf163990fe253fe1622883"
RECORD = {"val_loss": 3.295, "minutes": 14.2, "hardware": "8xH100", "date": "2026-09-05"}
BOS_ID = 50256
CANONICAL_VAL_TOKENS = 10_000_000
POLAR_COEFFS = (
    (8.156554524902461, -22.48329292557795, 15.878769915207462),
    (4.042929935166739, -2.808917465908714, 0.5000178451051316),
    (3.8916678022926607, -2.772484153217685, 0.5060648178503393),
    (3.285753657755655, -2.3681294933425376, 0.46449024233003106),
    (2.3465413258596377, -1.7097828382687081, 0.42323551169305323),
)


@dataclass
class TinyConfig:
    n_layer: int = 6
    n_embd: int = 384
    n_head: int = 6
    sequence_len: int = 512
    vocab_size: int = 50257
    dropout: float = 0.1
    window_pattern: str = "SSSL"
    xsa_mode: str = "all"
    num_iterations: int = 2
    iteration_transition_ratio: float = 0.3
    train_backprop_iterations: int = 1
    mtp_predict: int = 3
    mtp_anneal_frac: float = 0.66
    device_batch_size: int = 2
    effective_tokens: int = 65536
    train_tokens: int = 1_000_000
    val_tokens: int = 100_000
    num_epochs: int = 15
    max_steps: int = 0
    matrix_lr: float = 0.04
    scalar_lr: float = 0.25
    embedding_lr: float = 0.15
    unembedding_lr: float = 0.001
    lr_multiplier: float = 0.8
    weight_decay: float = 0.8
    wd_phase1_epoch: float = 2
    wd_phase2_epoch: float = 8
    wd_mid: float = 0.1
    wd_end: float = 0.93
    warmdown_ratio: float = 0.4
    swa_last_epochs: int = 4
    update_ema_every: int = 10
    ema_decay_per_epoch: float = 0.15
    mg_every: int = 1
    mg_step_norm: float = 0.5
    muon_eq_r: bool = False
    precision: str = "auto"
    device: str = "auto"
    use_compile: bool = False
    ema_device: str = "cpu"
    seed: int = 42
    eval_batches: int = 0
    run_dir: str = "runs/tiny_current_colab"

    def validate(self):
        for name in ("n_layer", "n_embd", "n_head", "sequence_len", "vocab_size",
                     "device_batch_size", "effective_tokens", "num_epochs", "train_tokens", "val_tokens"):
            if getattr(self, name) < 1:
                raise ValueError(f"{name} must be positive")
        if self.n_embd < 32 or self.n_embd % self.n_head or (self.n_embd // self.n_head) % 4:
            raise ValueError("Require width >= 32 and a head dimension divisible by 4")
        if self.effective_tokens % (self.device_batch_size * self.sequence_len):
            raise ValueError("effective_tokens must divide into whole microbatches")
        if self.xsa_mode not in ("off", "first6", "all"):
            raise ValueError("xsa_mode must be off, first6 or all")
        if not self.window_pattern or set(self.window_pattern) - set("SL"):
            raise ValueError("window_pattern uses S and L")
        if not 1 <= self.mtp_predict <= 4 or self.num_iterations < 1 or self.train_backprop_iterations < 1:
            raise ValueError("Invalid MTP / recurrence count")
        if not 0 <= self.iteration_transition_ratio <= 1 or not 0 < self.mtp_anneal_frac <= 1:
            raise ValueError("Invalid schedule fractions")
        if not 0 <= self.warmdown_ratio <= 1 or not 0 <= self.dropout < 1:
            raise ValueError("Invalid warmdown / dropout")
        if not 0 < self.ema_decay_per_epoch < 1 or self.update_ema_every < 0:
            raise ValueError("Invalid EMA settings")
        if self.wd_phase1_epoch < 0 or self.wd_phase2_epoch <= self.wd_phase1_epoch:
            raise ValueError("WD phases must increase")
        if self.max_steps < 0 or self.eval_batches < 0 or self.mg_every < 0 or self.swa_last_epochs < 0:
            raise ValueError("Counts must be nonnegative")
        if self.mg_every and self.effective_tokens // self.sequence_len < 2:
            raise ValueError("Meta-gradient requires at least two sequences per effective batch")
        return self


def preset(name="colab_t4", **overrides):
    cfg = TinyConfig()
    if name == "smoke":
        cfg = replace(cfg, n_layer=4, n_embd=128, n_head=4, sequence_len=128,
                      effective_tokens=2048, train_tokens=32768, val_tokens=8192,
                      num_epochs=3, update_ema_every=2, swa_last_epochs=2)
    elif name == "colab_l4":
        cfg = replace(cfg, n_layer=8, n_embd=512, n_head=8, device_batch_size=4)
    elif name == "colab_a100":
        cfg = replace(cfg, n_layer=10, n_embd=640, n_head=10, sequence_len=1024,
                      device_batch_size=4)
    elif name == "upstream_shape":
        cfg = replace(cfg, n_layer=16, n_embd=1024, n_head=8, sequence_len=2048,
                      device_batch_size=32, effective_tokens=524288,
                      train_tokens=100_000_000, val_tokens=10_000_000)
    elif name != "colab_t4":
        raise ValueError(f"Unknown preset {name}")
    return replace(cfg, **overrides).validate()


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_device(cfg):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if cfg.device == "auto" else torch.device(cfg.device)
    if cfg.precision == "auto":
        dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
    else:
        dtype = {"fp32": torch.float32, "bf16": torch.bfloat16, "fp16": torch.float16}[cfg.precision]
    if dtype == torch.bfloat16 and device.type == "cuda" and not torch.cuda.is_bf16_supported():
        raise ValueError("This GPU has no native BF16. Use auto / fp32, or explicitly try fp16.")
    if dtype != torch.float32 and device.type != "cuda":
        raise ValueError("Portable mixed precision is supported on CUDA only")
    return device, dtype


def amp_context(device, dtype):
    return torch.autocast("cuda", dtype=dtype) if device.type == "cuda" and dtype != torch.float32 else nullcontext()


def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def norm(x):
    if hasattr(F, "rms_norm"):
        return F.rms_norm(x, (x.size(-1),))
    return (x.float() * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + torch.finfo(x.dtype).eps)).to(x.dtype)


def apply_rotary(x, cos, sin):
    left, right = x.chunk(2, dim=-1)
    return torch.cat((left * cos + right * sin, right * cos - left * sin), dim=-1)


class Attention(nn.Module):
    def __init__(self, cfg, index):
        super().__init__()
        self.heads, self.dim = cfg.n_head, cfg.n_embd // cfg.n_head
        self.c_q = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.c_k = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.c_v = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.ve_gate = nn.Linear(32, cfg.n_head, bias=False) if index % 2 == (cfg.n_layer - 1) % 2 else None
        self.attn_gate = nn.Linear(12, cfg.n_head, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)
        long = cfg.window_pattern[index % len(cfg.window_pattern)] == "L" or index == cfg.n_layer - 1
        self.window = cfg.sequence_len if long else cfg.sequence_len // 2
        self.key_offset = long

    def forward(self, x, ve, cos, sin, alpha):
        b, t, c = x.shape
        q, k, v = [layer(x).view(b, t, self.heads, self.dim) for layer in (self.c_q, self.c_k, self.c_v)]
        if ve is not None:
            gate = 2 * torch.sigmoid(self.ve_gate(x[..., :32]))
            v = v + gate.unsqueeze(-1) * ve.view(b, t, self.heads, self.dim)
        cos, sin = cos.to(q.dtype), sin.to(q.dtype)
        q, k = norm(apply_rotary(q, cos, sin)), norm(apply_rotary(k, cos, sin))
        if self.key_offset and t > 1:
            stationary = torch.cat((k[:, :1, :, self.dim // 2:], k[:, :-1, :, self.dim // 2:]), dim=1)
            k = torch.cat((k[..., :self.dim // 2], stationary), dim=-1)
        if self.window >= t:
            y = F.scaled_dot_product_attention(q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), is_causal=True)
        else:
            positions = torch.arange(t, device=x.device)
            distance = positions[:, None] - positions[None, :]
            mask = (distance >= 0) & (distance <= self.window)
            y = F.scaled_dot_product_attention(q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), attn_mask=mask)
        y = y.transpose(1, 2)
        if alpha is not None:
            unit_v = v / v.square().sum(-1, keepdim=True).sqrt().clamp_min(1e-4)
            coefficient = (y * unit_v).sum(-1, keepdim=True) * alpha.tanh().to(y.dtype).view(1, 1, self.heads, 1)
            y = y - coefficient * unit_v
        y = y * torch.sigmoid(self.attn_gate(x[..., :12])).unsqueeze(-1)
        return self.dropout(self.c_proj(y.contiguous().view(b, t, c)))


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        hidden = 256 * ((8 * cfg.n_embd // 3 + 255) // 256)
        self.c_gate = nn.Linear(cfg.n_embd, hidden, bias=False)
        self.c_fc = nn.Linear(cfg.n_embd, hidden, bias=False)
        self.c_proj = nn.Linear(hidden, cfg.n_embd, bias=False)

    def forward(self, x):
        return self.c_proj(F.silu(self.c_gate(x)) * self.c_fc(x))


class Block(nn.Module):
    def __init__(self, cfg, index):
        super().__init__()
        self.attn, self.mlp = Attention(cfg, index), MLP(cfg)

    def forward(self, x, ve, cos, sin, alpha):
        x = x + self.attn(norm(x), ve, cos, sin, alpha)
        return x + self.mlp(norm(x))


class LinearT(nn.Module):
    def __init__(self, width, vocab):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(width, vocab))

    def forward(self, x):
        return x @ self.weight.to(x.dtype)


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.config = cfg.validate()
        padded = 64 * math.ceil(cfg.vocab_size / 64)
        self.wte = nn.Embedding(padded, cfg.n_embd)
        self.blocks = nn.ModuleList(Block(cfg, i) for i in range(cfg.n_layer))
        self.lm_head = LinearT(cfg.n_embd, padded)
        self.resid_lambdas = nn.Parameter(torch.ones(cfg.n_layer))
        self.x0_lambdas = nn.Parameter(torch.zeros(cfg.n_layer))
        self.skip_weights = nn.Parameter(torch.ones(cfg.n_layer // 2))
        self.xsa_alphas = nn.Parameter(torch.zeros(cfg.n_layer, cfg.n_head), requires_grad=cfg.xsa_mode != "off")
        self.ve_projs = nn.ModuleDict({str(i): nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
                                    for i in range(cfg.n_layer) if i % 2 == (cfg.n_layer - 1) % 2})
        dim = cfg.n_embd // cfg.n_head
        half = dim // 4
        inv = 1 / (10000 ** (torch.arange(0, half * 2, 2).float() / (half * 2)))
        inv = torch.cat((inv, torch.zeros(dim // 2 - half)))
        angles = torch.outer(torch.arange(cfg.sequence_len).float(), inv)
        self.register_buffer("cos", angles.cos()[None, :, None, :], persistent=False)
        self.register_buffer("sin", angles.sin()[None, :, None, :], persistent=False)
        self.init_weights()

    @torch.no_grad()
    def init_weights(self):
        nn.init.normal_(self.wte.weight, std=1)
        nn.init.normal_(self.lm_head.weight, std=0.001)
        scale = math.sqrt(3 / self.config.n_embd)
        for block in self.blocks:
            for layer in (block.attn.c_q, block.attn.c_k, block.attn.c_v, block.mlp.c_gate, block.mlp.c_fc):
                nn.init.uniform_(layer.weight, -scale, scale)
            nn.init.zeros_(block.attn.c_proj.weight)
            nn.init.zeros_(block.mlp.c_proj.weight)
            nn.init.zeros_(block.attn.attn_gate.weight)
            if block.attn.ve_gate is not None:
                nn.init.zeros_(block.attn.ve_gate.weight)
        for layer in self.ve_projs.values():
            nn.init.uniform_(layer.weight, -scale, scale)
        self.resid_lambdas.fill_(1.1)
        self.x0_lambdas.fill_(0.1)
        self.skip_weights.fill_(1)
        self.xsa_alphas.zero_()

    def _pass(self, x, anchor, cos, sin):
        skips, encoder_layers = [], self.config.n_layer // 2
        for i, block in enumerate(self.blocks):
            if i >= encoder_layers and skips:
                x = x + self.skip_weights[i - encoder_layers] * skips.pop()
            x = self.resid_lambdas[i] * x + self.x0_lambdas[i] * anchor
            ve = self.ve_projs[str(i)](anchor) if str(i) in self.ve_projs else None
            use_xsa = self.config.xsa_mode == "all" or (self.config.xsa_mode == "first6" and i < 6)
            x = block(x, ve, cos, sin, self.xsa_alphas[i] if use_xsa else None)
            if i < encoder_layers:
                skips.append(x)
        return x

    def forward(self, idx, targets=None, *, iterations=1, mtp_weights=None, reduction="mean"):
        if not 1 <= iterations <= self.config.num_iterations or idx.size(1) > self.config.sequence_len:
            raise ValueError("Invalid recurrence / sequence length")
        x = norm(self.wte(idx))
        anchor = x
        cos, sin = self.cos[:, :idx.size(1)], self.sin[:, :idx.size(1)]
        grad_start = max(0, iterations - self.config.train_backprop_iterations) if self.training else 0
        for i in range(iterations):
            if self.training and i < grad_start:
                with torch.no_grad():
                    x = norm(self._pass(x, anchor, cos, sin))
                x = x.detach()
            else:
                x = norm(self._pass(x, anchor, cos, sin))
        h = norm(x)
        raw_logits = self.lm_head(h).float()
        logits = 15 * torch.tanh(raw_logits / 15)
        if targets is None:
            return logits[..., :self.config.vocab_size]
        if self.training and mtp_weights is not None:
            return shared_mtp_loss(logits, targets, mtp_weights, reduction)
        logits = logits[..., :self.config.vocab_size]
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), reduction="none")
        return loss if reduction == "none" else loss.mean()


def shared_mtp_loss(logits, targets, weights, reduction="mean"):
    # Match the upstream kernel's flattened offset convention, including row boundaries.
    flat = logits.reshape(-1, logits.size(-1))
    target = targets.reshape(-1)
    lse = flat.logsumexp(-1)
    loss = torch.zeros_like(lse)
    for offset, weight in enumerate(weights):
        count = target.numel() - offset
        if count <= 0:
            break
        selected = flat[:count].gather(1, target[offset:, None]).squeeze(1)
        loss = loss + F.pad((lse[:count] - selected) * weight, (0, offset))
    return loss if reduction == "none" else loss.mean()


def iteration_count(cfg, step, total):
    if cfg.num_iterations == 1:
        return 1
    return cfg.num_iterations if step / max(total, 1) >= 1 - cfg.iteration_transition_ratio else 1


def mtp_schedule(cfg, step, total):
    initial, result = (1.0, 0.5, 0.25, 0.125), [1.0]
    fraction, n = step / max(1, total), cfg.mtp_predict
    for k in range(1, n):
        start = (n - 1 - k) / (n - 1) * cfg.mtp_anneal_frac
        end = (n - k) / (n - 1) * cfg.mtp_anneal_frac
        result.append(initial[k] * min(1.0, max(0.0, (end - fraction) / (end - start))))
    return result


def schedule(cfg, step, total, steps_per_epoch):
    warmdown = round(cfg.warmdown_ratio * total)
    lr_mult = 1.0 if warmdown <= 0 or step <= total - warmdown else max(0.0, (total - step) / warmdown)
    swa_start = total - min(cfg.swa_last_epochs, cfg.num_epochs) * steps_per_epoch
    if cfg.swa_last_epochs and step >= swa_start:
        pos = (step - swa_start) % steps_per_epoch
        lr_mult = 0.05 + (max(lr_mult, 0.05) - 0.05) * (1 + math.cos(math.pi * pos / steps_per_epoch)) / 2
    epoch = step / steps_per_epoch
    # Preserve absolute WD phase epochs; short smoke runs need not reach the final ramp.
    if epoch <= cfg.wd_phase1_epoch:
        wd = cfg.weight_decay
    elif epoch <= cfg.wd_phase2_epoch:
        mix = (epoch - cfg.wd_phase1_epoch) / (cfg.wd_phase2_epoch - cfg.wd_phase1_epoch)
        wd = cfg.weight_decay + mix * (cfg.wd_mid - cfg.weight_decay)
    else:
        mix = min(1, (epoch - cfg.wd_phase2_epoch) / max(1e-9, cfg.num_epochs - cfg.wd_phase2_epoch))
        wd = cfg.wd_mid + mix * (cfg.wd_end - cfg.wd_mid)
    return lr_mult, wd


def muon_direction(grads, momentum_buffer, variance_buffer, momentum, row_equilibrate=False):
    momentum_buffer.lerp_(grads, 1 - momentum)
    g = grads.lerp(momentum_buffer, momentum)
    if row_equilibrate:
        g = g / g.float().norm(dim=-1, keepdim=True).clamp_min(1e-7).to(g.dtype)
    native_bf16 = g.device.type == "cuda" and torch.cuda.is_bf16_supported()
    x = g.to(torch.bfloat16 if native_bf16 else torch.float32)
    x = x / (x.norm(dim=(-2, -1), keepdim=True) * 1.02 + 1e-6)
    for a, b, c in POLAR_COEFFS:
        if x.size(-2) > x.size(-1):
            gram = x.mT @ x
            x = a * x + x @ (b * gram + c * (gram @ gram))
        else:
            gram = x @ x.mT
            x = a * x + (b * gram + c * (gram @ gram)) @ x
    g = x.float()
    dim = -1 if g.size(-2) >= g.size(-1) else -2
    v = g.square().mean(dim=dim, keepdim=True)
    count = g.size(dim)
    old_norm = (v.sum((-2, -1), keepdim=True) * count).sqrt()
    variance_buffer.lerp_(v, 0.05)
    scale = variance_buffer.clamp_min(1e-10).rsqrt()
    new_norm = (v * count * scale.square()).sum((-2, -1), keepdim=True).sqrt()
    return g * scale * old_norm / new_norm.clamp_min(1e-10)


class MuonAdamW(torch.optim.Optimizer):
    """Single-device version; FP32 states, no mandatory NCCL or fused compilation."""

    def __init__(self, groups, cfg):
        super().__init__(groups, {})
        self.cfg = cfg

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            active = [p for p in group["params"] if p.grad is not None]
            if not active:
                continue
            lr, wd = group["lr"], group["weight_decay"]
            if group["kind"] == "adamw":
                beta1, beta2 = group["betas"]
                for p in active:
                    state = self.state[p]
                    if not state:
                        state.update(step=0, m=torch.zeros_like(p), v=torch.zeros_like(p))
                    state["step"] += 1
                    state["m"].lerp_(p.grad, 1 - beta1)
                    state["v"].lerp_(p.grad.square(), 1 - beta2)
                    denom = (state["v"] / (1 - beta2 ** state["step"])).sqrt().add_(1e-10)
                    p.mul_(1 - lr * wd)
                    p.addcdiv_(state["m"], denom, value=-lr / (1 - beta1 ** state["step"]))
            else:
                # State belongs to each parameter, so unused parameters are never decayed.
                for p in active:
                    state = self.state[p]
                    if not state:
                        dim = -1 if p.size(-2) >= p.size(-1) else -2
                        shape = list(p.shape)
                        shape[dim] = 1
                        state.update(momentum=torch.zeros_like(p), variance=torch.zeros(shape, device=p.device))
                    g = muon_direction(p.grad, state["momentum"], state["variance"], group["momentum"], self.cfg.muon_eq_r)
                    rate = lr * max(1, p.size(-2) / p.size(-1)) ** 0.5
                    cautious = (g * p) >= 0
                    p.sub_(rate * g + rate * wd * p * cautious)


def build_optimizer(model, cfg):
    gates = [b.attn.attn_gate.weight for b in model.blocks]
    gate_ids = {id(p) for p in gates}
    matrices = [p for p in list(model.blocks.parameters()) + list(model.ve_projs.parameters()) if id(p) not in gate_ids]
    groups = []
    def add(kind, params, lr, betas=(0.8, 0.95), wd=0):
        params = [p for p in params if p.requires_grad]
        if params:
            groups.append(dict(kind=kind, params=params, lr=lr * cfg.lr_multiplier,
                               initial_lr=lr * cfg.lr_multiplier, weight_decay=wd,
                               initial_wd=wd, betas=betas, momentum=0.95))
    add("adamw", model.lm_head.parameters(), cfg.unembedding_lr, wd=cfg.weight_decay)
    add("adamw", model.wte.parameters(), cfg.embedding_lr, wd=cfg.weight_decay)
    add("adamw", [model.resid_lambdas], cfg.scalar_lr * 0.01)
    add("adamw", [model.x0_lambdas], cfg.scalar_lr, (0.96, 0.95))
    add("adamw", [model.skip_weights], cfg.scalar_lr * 0.01)
    add("adamw", gates, cfg.scalar_lr, (0.9, 0.99))
    add("adamw", [model.xsa_alphas], cfg.scalar_lr, (0.9, 0.95))
    for shape in sorted({p.shape for p in matrices}):
        add("muon", [p for p in matrices if p.shape == shape], cfg.matrix_lr, wd=cfg.weight_decay)
    assigned = [id(p) for group in groups for p in group["params"]]
    expected = {id(p) for p in model.parameters() if p.requires_grad}
    if len(assigned) != len(set(assigned)) or set(assigned) != expected:
        raise RuntimeError("Optimizer parameters are duplicated or missing")
    return MuonAdamW(groups, cfg)


def module_summary(model):
    rows = []
    for name, child in model.named_children():
        count = sum(p.numel() for p in child.parameters())
        rows.append({"module": name, "parameters": count})
    rows.append({"module": "top_level_scalars", "parameters": sum(p.numel() for p in model.parameters(recurse=False))})
    for row in rows:
        print(f"{row['module']:<22} {row['parameters']:>14,}")
    total = sum(p.numel() for p in model.parameters())
    print(f"{'TOTAL':<22} {total:>14,}")
    return rows


def load_tensors(path):
    return torch.load(path, map_location="cpu", weights_only=True)


class PackedData:
    def __init__(self, data, cfg, shuffle):
        self.data = load_tensors(data) if isinstance(data, (str, Path)) else data
        self.cfg, self.shuffle = cfg, shuffle
        self.tokens = self.data["tokens"].long()
        self.starts = self.data["doc_starts"].long()
        if not self.starts.numel() or self.starts[0] != 0 or self.tokens.numel() < cfg.device_batch_size * (cfg.sequence_len + 1):
            raise ValueError("Not enough valid data for one complete microbatch")
        if self.tokens.min() < 0 or self.tokens.max() >= cfg.vocab_size:
            raise ValueError("Token IDs do not match the configured vocabulary")
        if self.starts[-1] >= self.tokens.numel() or torch.any(self.starts[1:] <= self.starts[:-1]):
            raise ValueError("Document starts must be strictly increasing and in range")
        if cfg.vocab_size == 50257 and int(self.data.get("bos_id", -1)) != BOS_ID:
            raise ValueError("Expected GPT-2 BOS/EOT metadata")
        self.docs = self.tokens.tensor_split(self.starts[1:].tolist())
        self.order = list(range(len(self.docs)))
        self.next_epoch = 1
        self.num_batches = self.tokens.numel() // (cfg.sequence_len + 1) // cfg.device_batch_size
        self.tokens_per_epoch = self.num_batches * cfg.device_batch_size * cfg.sequence_len

    def batches(self, epoch=1):
        if self.shuffle:
            if epoch != self.next_epoch:
                raise ValueError("Training epochs must be visited in order")
            if epoch > 1:
                permutation = torch.randperm(len(self.docs), generator=torch.Generator().manual_seed(epoch))
                self.order = [self.order[i] for i in permutation.tolist()]
            self.next_epoch += 1
        tokens = torch.cat([self.docs[i] for i in self.order]) if self.shuffle else self.tokens
        count = tokens.numel() // (self.cfg.sequence_len + 1)
        seqs = tokens[:count * (self.cfg.sequence_len + 1)].view(count, self.cfg.sequence_len + 1)
        if self.shuffle:
            perm = torch.randperm(count, generator=torch.Generator().manual_seed(epoch + 1000))
        else:
            perm = torch.from_numpy(np.random.RandomState(int(self.data.get("seq_shuffle_seed", 42))).permutation(count))
        for start in range(0, self.num_batches * self.cfg.device_batch_size, self.cfg.device_batch_size):
            batch = seqs[perm[start:start + self.cfg.device_batch_size]]
            yield batch[:, :-1].contiguous(), batch[:, 1:].contiguous()


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def prepare_data(cfg, root="fineweb_data_current", *, dataset_revision=None):
    """Study cache; reserves the full official validation prefix even for small runs."""
    import tiktoken
    from datasets import load_dataset
    from tqdm.auto import tqdm
    if cfg.val_tokens > CANONICAL_VAL_TOKENS or cfg.train_tokens > 100_000_000:
        raise ValueError("Study data must remain within the official token budget")
    encoder = tiktoken.get_encoding("gpt2")
    spec = dict(dataset="HuggingFaceFW/fineweb", name="sample-10BT", split="train",
                revision=dataset_revision, tokenizer="gpt2", validation_prefix=CANONICAL_VAL_TOKENS,
                train_tokens=cfg.train_tokens, val_tokens=cfg.val_tokens, storage_dtype="int32", cache_version=3)
    key = hashlib.sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest()[:16]
    directory = Path(root) / key
    manifest_path = directory / "manifest.json"
    paths = [directory / "train.pt", directory / "val.pt"]
    if manifest_path.exists() and all(p.exists() for p in paths):
        manifest = json.loads(manifest_path.read_text())
        if manifest.get("spec") == spec and all(sha256_file(p) == manifest["sha256"][p.name] for p in paths):
            print("Reusing verified cache:", directory)
            return tuple(paths)
    directory.mkdir(parents=True, exist_ok=True)
    kwargs = dict(name="sample-10BT", split="train", streaming=True)
    if dataset_revision:
        kwargs["revision"] = dataset_revision
    dataset = load_dataset("HuggingFaceFW/fineweb", **kwargs)
    iterator = iter(dataset)
    def collect(budget, keep, label):
        # Preallocation avoids a 100M-element Python-int list on Colab.
        buf, starts, consumed, stored = np.empty(keep, dtype=np.uint16), [], 0, 0
        bar = tqdm(total=budget, desc=label, unit="tok")
        for doc in iterator:
            ids = [BOS_ID] + encoder.encode_ordinary(doc["text"])
            take = min(len(ids), budget - consumed)
            save = min(take, max(0, keep - stored))
            if save:
                starts.append(stored)
                buf[stored:stored + save] = ids[:save]
                stored += save
            consumed += take
            bar.update(take)
            if consumed == budget:
                break
        bar.close()
        if consumed != budget or stored != keep:
            raise ValueError("Dataset ended before the requested token budget")
        # Older PyTorch cannot serialize uint16; keep a compact, portable int32 cache.
        return dict(tokens=torch.from_numpy(buf.astype(np.int32)), doc_starts=torch.tensor(starts), bos_id=BOS_ID,
                    seq_shuffle_seed=42 if label == "val prefix" else 43, seq_size=2049)
    # Train always begins after the SAME 10M-token document traversal as upstream.
    validation = collect(CANONICAL_VAL_TOKENS, cfg.val_tokens, "val prefix")
    training = collect(cfg.train_tokens, cfg.train_tokens, "train")
    for path, content in zip(paths, (training, validation)):
        torch.save(content, path)
    manifest = dict(spec=spec, sha256={p.name: sha256_file(p) for p in paths}, upstream=UPSTREAM_COMMIT)
    manifest_path.write_text(json.dumps(manifest, indent=2))
    del iterator, dataset
    return tuple(paths)


def split_episode(batches):
    total = sum(x.size(0) for x, _ in batches)
    cut, seen, support, query = total // 2, 0, [], []
    for x, y in batches:
        n = max(0, min(x.size(0), cut - seen))
        if n:
            support.append((x[:n], y[:n]))
        if n < x.size(0):
            query.append((x[n:], y[n:]))
        seen += x.size(0)
    return support, query


@contextmanager
def temporary_adaptation(params, distance):
    active = [p for p in params if p.grad is not None]
    if not active or distance == 0:
        yield False
        return
    length = torch.stack([p.grad.float().norm() for p in active]).norm()
    if not torch.isfinite(length) or length <= 1e-12:
        yield False
        return
    # Save exact FP32 weights so restores also remain exact when FP16 AMP overflows.
    originals = [p.detach().clone() for p in active]
    try:
        with torch.no_grad():
            for p in active:
                p.add_(p.grad, alpha=-distance / length.item())
        yield True
    finally:
        with torch.no_grad():
            for p, original in zip(active, originals):
                p.copy_(original)


def train_update(model, raw_model, optimizer, batches, cfg, step, total, device, dtype, scaler):
    optimizer.zero_grad(set_to_none=True)
    count = sum(x.numel() for x, _ in batches)
    iterations, weights = iteration_count(cfg, step, total), mtp_schedule(cfg, step, total)
    logged_loss = 0.0
    def backward(parts):
        nonlocal logged_loss
        for x_cpu, y_cpu in parts:
            x, y = x_cpu.to(device), y_cpu.to(device)
            with amp_context(device, dtype):
                loss = model(x, y, iterations=iterations, mtp_weights=weights)
            fraction = x.numel() / count
            scaler.scale(loss * fraction).backward()
            logged_loss += loss.detach().item() * fraction
    mg_applied = False
    if cfg.mg_every and step % cfg.mg_every == 0 and sum(x.size(0) for x, _ in batches) >= 2:
        support, query = split_episode(batches)
        backward(support)
        plastic = [p for block in raw_model.blocks for p in block.mlp.parameters()]
        with temporary_adaptation(plastic, cfg.mg_step_norm) as mg_applied:
            backward(query)
    else:
        backward(batches)
    scaler.unscale_(optimizer)
    old_scale = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    return dict(train_objective=logged_loss, processed_tokens=count, iterations=iterations,
                mg_applied=mg_applied, mtp_weights=weights,
                optimizer_step_applied=scaler.get_scale() >= old_scale)


@torch.no_grad()
def evaluate(model, data, byte_counts, cfg, device, dtype, iterations):
    was_training = model.training
    model.eval()
    loss_sum = nats = 0.0
    token_count = byte_count = 0
    try:
        for i, (x, y) in enumerate(data.batches()):
            if cfg.eval_batches and i >= cfg.eval_batches:
                break
            x, y = x.to(device), y.to(device)
            with amp_context(device, dtype):
                losses = model(x, y, iterations=iterations, reduction="none")
            counts = byte_counts.to(device)[y.reshape(-1)]
            loss_sum += losses.sum().item()
            token_count += y.numel()
            nats += losses[counts > 0].sum().item()
            byte_count += counts.sum().item()
    finally:
        model.train(was_training)
    if token_count == 0 or byte_count == 0:
        raise ValueError("Validation must contain nonempty text bytes")
    return dict(val_loss=loss_sum / token_count, val_bpb=nats / (math.log(2) * byte_count),
                evaluated_tokens=token_count, evaluated_bytes=byte_count, iterations=iterations)


class ParameterEMA:
    def __init__(self, model, beta, device="cpu"):
        self.beta, self.updates = beta, 0
        self.values = {n: torch.zeros_like(p, device=device, dtype=torch.float32) for n, p in model.named_parameters()}

    @torch.no_grad()
    def update(self, model):
        for name, p in model.named_parameters():
            value = self.values[name]
            value.lerp_(p.detach().to(value.device, torch.float32), 1 - self.beta)
        self.updates += 1

    def corrected(self):
        if not self.updates:
            return None
        return {n: value / (1 - self.beta ** self.updates) for n, value in self.values.items()}


def parameter_snapshot(model):
    return {n: p.detach().float().cpu().clone() for n, p in model.named_parameters()}


@torch.no_grad()
def load_parameters(model, values):
    for name, p in model.named_parameters():
        p.copy_(values[name].to(p.device, p.dtype))


def average_checkpoints(paths):
    weights = list(range(1, len(paths) + 1))
    average = None
    for path, weight in zip(paths, weights):
        state = load_tensors(path)
        if average is None:
            average = {n: p.float() * (weight / sum(weights)) for n, p in state.items()}
        else:
            for name in average:
                average[name].add_(state[name].float(), alpha=weight / sum(weights))
    return average


def train(cfg, train_path, val_path, byte_counts):
    cfg.validate()
    seed_everything(cfg.seed)
    device, dtype = resolve_device(cfg)
    directory = Path(cfg.run_dir)
    if (directory / "result.json").exists() or (directory / "config.json").exists():
        raise FileExistsError(f"Choose a new run_dir; {directory} already contains a run")
    directory.mkdir(parents=True, exist_ok=True)
    (directory / "config.json").write_text(json.dumps(asdict(cfg), indent=2))
    raw_model = GPT(cfg).to(device)
    module_summary(raw_model)
    optimizer = build_optimizer(raw_model, cfg)
    model = torch.compile(raw_model, dynamic=False) if cfg.use_compile else raw_model
    train_data, val_data = PackedData(train_path, cfg, True), PackedData(val_path, cfg, False)
    accum = cfg.effective_tokens // (cfg.device_batch_size * cfg.sequence_len)
    per_epoch = math.ceil(train_data.num_batches / accum)
    total = per_epoch * cfg.num_epochs
    target = min(total, cfg.max_steps) if cfg.max_steps else total
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda" and dtype == torch.float16)
    beta = cfg.ema_decay_per_epoch ** (cfg.update_ema_every / per_epoch) if cfg.update_ema_every else 0
    ema = ParameterEMA(raw_model, beta, cfg.ema_device) if cfg.update_ema_every else None
    print(f"{device} / {dtype}; microbatch={cfg.device_batch_size * cfg.sequence_len:,} tokens; accum={accum}")
    print(f"{target} planned updates; smaller final updates count actual tokens. Not a leaderboard run.")
    initial = evaluate(model, val_data, byte_counts, cfg, device, dtype, cfg.num_iterations)
    print("Initial clean validation:", initial)
    history, validation, times, late_paths = [], [], [], []
    processed = 0
    step = 0
    best_loss = float("inf")
    best_kind = None
    start = time.perf_counter()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    def assess(kind, epoch):
        nonlocal best_loss, best_kind
        metrics = evaluate(model, val_data, byte_counts, cfg, device, dtype, cfg.num_iterations)
        entry = dict(kind=kind, epoch=epoch, step=step, processed_tokens=processed, **metrics)
        validation.append(entry)
        print(f"{kind}: CE={metrics['val_loss']:.5f}, BPB={metrics['val_bpb']:.5f}")
        if metrics["val_loss"] < best_loss:
            best_loss, best_kind = metrics["val_loss"], kind
            torch.save(dict(config=asdict(cfg), parameters=parameter_snapshot(raw_model),
                            upstream=UPSTREAM_COMMIT, metrics=entry), directory / "best.pt")
        return entry
    for epoch in range(1, cfg.num_epochs + 1):
        batches = iter(train_data.batches(epoch))
        while step < target:
            group = []
            for _ in range(accum):
                pair = next(batches, None)
                if pair is None:
                    break
                group.append(pair)
            if not group:
                break
            lr_mult, wd = schedule(cfg, step, total, per_epoch)
            for param_group in optimizer.param_groups:
                param_group["lr"] = param_group["initial_lr"] * lr_mult
                param_group["weight_decay"] = (wd if param_group["initial_wd"] > 0 else 0.0)
                param_group["momentum"] = 0.85 + min(step / 300, 1) * 0.1
            model.train()
            synchronize(device)
            tick = time.perf_counter()
            entry = train_update(model, raw_model, optimizer, group, cfg, step, total, device, dtype, scaler)
            step += 1
            if ema is not None and step % cfg.update_ema_every == 0:
                ema.update(raw_model)
            synchronize(device)
            seconds = time.perf_counter() - tick
            times.append(seconds)
            processed += entry["processed_tokens"]
            entry.update(step=step, epoch=epoch, seconds=seconds, lr_multiplier=lr_mult,
                         weight_decay=wd, tokens_per_second=entry["processed_tokens"] / seconds)
            history.append(entry)
            if step == 1 or step % 10 == 0 or step == target:
                print(f"step {step}/{target} objective={entry['train_objective']:.4f} "
                      f"{entry['tokens_per_second']:,.0f} tok/s, recurrence={entry['iterations']}, MG={entry['mg_applied']}")
        assess("live", epoch)
        if cfg.swa_last_epochs and epoch > cfg.num_epochs - min(cfg.swa_last_epochs, cfg.num_epochs):
            path = directory / f"late_epoch_{epoch:03d}.pt"
            torch.save(parameter_snapshot(raw_model), path)
            late_paths.append(path)
        if step >= target:
            break
    live = parameter_snapshot(raw_model)
    if ema is not None and ema.updates:
        load_parameters(raw_model, ema.corrected())
        assess("ema", epoch)
    if len(late_paths) >= 2:
        load_parameters(raw_model, average_checkpoints(late_paths))
        assess("recency_weight_average", epoch)
    load_parameters(raw_model, live)
    elapsed = time.perf_counter() - start
    steady = history[1:] if len(history) > 1 else history
    rate = sum(h["processed_tokens"] for h in steady) / sum(h["seconds"] for h in steady)
    result = dict(upstream=UPSTREAM_COMMIT, checked="2026-10-19", mode="portable_study_not_record",
                  config=asdict(cfg), hardware=str(device), dtype=str(dtype), parameters=sum(p.numel() for p in raw_model.parameters()),
                  processed_tokens=processed, unique_source_tokens=int(train_data.tokens.numel()), steps=step,
                  optimizer_updates_applied=sum(h["optimizer_step_applied"] for h in history),
                  training_seconds=sum(times), wall_seconds=elapsed, steady_tokens_per_second=rate,
                  estimated_1000_full_updates_minutes=1000 * cfg.effective_tokens / rate / 60,
                  peak_allocated_gib=torch.cuda.max_memory_allocated(device) / 2**30 if device.type == "cuda" else None,
                  peak_reserved_gib=torch.cuda.max_memory_reserved(device) / 2**30 if device.type == "cuda" else None,
                  initial=initial, best_val_loss=best_loss, best_kind=best_kind, validation=validation,
                  data_sha256={"train": sha256_file(train_path), "val": sha256_file(val_path)})
    (directory / "result.json").write_text(json.dumps(result, indent=2))
    with (directory / "history.csv").open("w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(history[0]))
        writer.writeheader()
        writer.writerows(history)
    print(json.dumps({k: result[k] for k in ("best_val_loss", "best_kind", "processed_tokens", "wall_seconds", "estimated_1000_full_updates_minutes")}, indent=2))
    return raw_model, result, history


def token_byte_counts(encoder):
    return torch.tensor([0 if i == BOS_ID else len(encoder.decode_single_token_bytes(i))
                         for i in range(encoder.n_vocab)], dtype=torch.int64)


def probe(cfg, warmup=2, measured=5):
    if measured < 1 or warmup < 0:
        raise ValueError("Use nonnegative warmup and at least one measured update")
    cfg.validate()
    device, dtype = resolve_device(cfg)
    if device.type != "cuda":
        raise ValueError("Run the fit probe on the actual Colab CUDA GPU")
    seed_everything(cfg.seed)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)
    model = optimizer = runner = ema = None
    try:
        model = GPT(cfg).to(device)
        runner = torch.compile(model, dynamic=False) if cfg.use_compile else model
        optimizer = build_optimizer(model, cfg)
        ema = ParameterEMA(model, 0.9, cfg.ema_device) if cfg.update_ema_every else None
        accum = cfg.effective_tokens // (cfg.device_batch_size * cfg.sequence_len)
        batches = [(torch.randint(cfg.vocab_size, (cfg.device_batch_size, cfg.sequence_len)),
                    torch.randint(cfg.vocab_size, (cfg.device_batch_size, cfg.sequence_len))) for _ in range(accum)]
        scaler = torch.cuda.amp.GradScaler(enabled=dtype == torch.float16)
        durations = []
        for step in range(warmup + measured):
            synchronize(device)
            tick = time.perf_counter()
            # Exercise the early stage with all MTP offsets still active.
            train_update(runner, model, optimizer, batches, cfg, 0, 1, device, dtype, scaler)
            if ema is not None:
                ema.update(model)
            synchronize(device)
            if step >= warmup:
                durations.append(time.perf_counter() - tick)
        # The late recurrence stage is a separate graph; measure it explicitly as well.
        stage_cfg = replace(cfg, iteration_transition_ratio=1.0)
        late_durations = []
        for step in range(warmup + measured):
            synchronize(device)
            tick = time.perf_counter()
            train_update(runner, model, optimizer, batches, stage_cfg, 0, 1, device, dtype, scaler)
            if ema is not None:
                ema.update(model)
            synchronize(device)
            if step >= warmup:
                late_durations.append(time.perf_counter() - tick)
        worst = max(sum(durations) / measured, sum(late_durations) / measured)
        result = dict(parameters=sum(p.numel() for p in model.parameters()), dtype=str(dtype),
                      microbatch_tokens=cfg.device_batch_size * cfg.sequence_len,
                      accumulation=accum, effective_tokens=cfg.effective_tokens,
                      early_step_seconds=sum(durations) / measured,
                      late_step_seconds=sum(late_durations) / measured,
                      peak_allocated_gib=torch.cuda.max_memory_allocated(device) / 2**30,
                      peak_reserved_gib=torch.cuda.max_memory_reserved(device) / 2**30,
                      total_vram_gib=torch.cuda.get_device_properties(device).total_memory / 2**30,
                      conservative_tokens_per_second=cfg.effective_tokens / worst,
                      estimate_1000_updates_minutes=1000 * worst / 60)
        print(json.dumps(result, indent=2))
        return result
    except torch.cuda.OutOfMemoryError:
        print("OOM: reduce microbatch size, then sequence length/model size. Keep effective tokens divisible.")
        return None
    finally:
        del runner, model, optimizer, ema
        gc.collect()
        torch.cuda.empty_cache()


@torch.no_grad()
def generate(model, encoder, prompt, max_new_tokens=120, temperature=0.7, top_k=40):
    cfg = model.config
    device = next(model.parameters()).device
    was_training = model.training
    model.eval()
    ids = [BOS_ID] + encoder.encode_ordinary(prompt)
    try:
        for _ in range(max_new_tokens):
            x = torch.tensor([ids[-cfg.sequence_len:]], device=device)
            logits = model(x, iterations=cfg.num_iterations)[0, -1]
            logits[BOS_ID] = -float("inf")
            if temperature <= 0:
                token = logits.argmax().item()
            else:
                logits = logits / temperature
                if 0 < top_k < logits.numel():
                    logits[logits < logits.topk(top_k).values[-1]] = -float("inf")
                token = torch.multinomial(logits.softmax(-1), 1).item()
            ids.append(token)
    finally:
        model.train(was_training)
    return encoder.decode(ids[1:])


## Choose a study size

`smoke` checks the whole pipeline with a small model/data budget. `colab_t4`,
`colab_l4`, and `colab_a100` are **candidate shapes to probe, not guaranteed fits**.
They keep the current recipe and 15 epochs but reduce model/data size.
`upstream_shape` restores the full architecture and data sizes, but still uses
the portable backend; it is not the official launch path.

Effective tokens per update = sequences per microbatch x context tokens x
accumulation steps, on this single-device path. The loader's last update in
each epoch may contain fewer tokens, which are counted exactly. Replay across
epochs is processed exposure, not extra unique data.

Leave `MUON_EQ_R=False` for the control. The fixed official settings and why
Colab scores are not comparable are documented in the reference section.


In [ ]:
#@title 3. Configuration
from datetime import datetime, timezone
PRESET = "smoke" #@param ["smoke", "colab_t4", "colab_l4", "colab_a100", "upstream_shape"]
MUON_EQ_R = False #@param {type:"boolean"}
RUN_PROBE = True #@param {type:"boolean"}
RUN_TRAINING = True #@param {type:"boolean"}
RUN_SAMPLING = False #@param {type:"boolean"}
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
cfg = preset(PRESET, muon_eq_r=MUON_EQ_R, run_dir=f"runs/{PRESET}_{stamp}")
# Use replace(cfg, n_layer=..., n_embd=..., sequence_len=..., ...) for a controlled override.
cfg.validate()
print(json.dumps(asdict(cfg), indent=2))
device, dtype = resolve_device(cfg)
print("Selected:", device, dtype)
print("Accumulation steps:", cfg.effective_tokens // (cfg.device_batch_size * cfg.sequence_len))
print("This configuration is a portable study, not a leaderboard submission.")


In [ ]:
#@title 4. Parameter summary without allocating GPU parameters
with torch.device("meta"):
    summary_model = GPT(cfg)
summary_rows = module_summary(summary_model)
del summary_model


## Probe before training

The probe uses random tokens, so it measures fit and update cost, not learning.
It exercises both recurrence stages, shared-head MTP, meta-gradient restoration,
optimizer state and EMA updates. The late-stage test keeps auxiliary MTP active
for a conservative stress test, even though real late training anneals it away.

Compare allocated and reserved **GiB** with physical VRAM; keep headroom for
validation and the real runtime. Reduce `device_batch_size` first on OOM and
preserve divisibility of the effective token batch. A successful largest probe
establishes a lower bound on capacity, not the maximum possible model.


In [ ]:
#@title 5. Optional fit and timing probe
probe_result = None
if RUN_PROBE and torch.cuda.is_available():
    probe_result = probe(cfg, warmup=2, measured=3)
else:
    print("Probe skipped. Enable RUN_PROBE on a CUDA runtime to measure the actual GPU.")


## Data: preserve the train/validation boundary

FineWeb is encoded with GPT-2 BPE. Even a study run consumes the entire **10M-token
canonical validation prefix** before selecting training documents. It only stores
the smaller requested validation subset. This takes preprocessing time but avoids
training on examples reserved for official validation.

Preallocated arrays avoid a huge Python-int list. The cache manifest includes
source, tokenizer and budgets; cached bytes are checked against saved hashes.
This study cache is not asserted to equal upstream's canonical serialized files.
The official path below uses the exact preparation script and expected hashes.
An HF token is optional for public data; do not put secrets in notebook cells.


In [ ]:
#@title 6. Prepare verified study data
import tiktoken
DATA_DIR = "/content/fineweb_data_current" if "google.colab" in sys.modules else "fineweb_data_current"
train_path, val_path = prepare_data(cfg, DATA_DIR)
encoder = tiktoken.get_encoding("gpt2")
byte_counts = token_byte_counts(encoder)
print("Train:", train_path)
print("Validation:", val_path)


## Train and evaluate

`train_objective` includes weighted MTP terms and may include gradients from an
adapted point. **Validation always uses clean next-token CE/BPB**, with the
maximum recurrence count held fixed across candidates. Live, EMA and averaged
checkpoints are recorded separately; a smaller final batch is logged explicitly.

Files saved under `cfg.run_dir`: `config.json`, `result.json`, `history.csv`,
late parameter snapshots, and `best.pt` containing the selected parameters and
their validation metrics. Re-run Configuration to get a new run directory before
repeating training. No earlier artifacts are silently overwritten.


In [ ]:
#@title 7. Training
model = result = history = None
if RUN_TRAINING:
    model, result, history = train(cfg, train_path, val_path, byte_counts)
else:
    print("Training disabled by RUN_TRAINING.")


In [ ]:
#@title 8. Inspect learning and timing
import matplotlib.pyplot as plt
if result is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot([row["step"] for row in history], [row["train_objective"] for row in history])
    axes[0].set(title="Training objective (includes MTP)", xlabel="Update attempt", ylabel="Objective")
    for kind in ("live", "ema", "recency_weight_average"):
        rows = [row for row in result["validation"] if row["kind"] == kind]
        if rows:
            axes[1].plot([row["processed_tokens"] for row in rows], [row["val_bpb"] for row in rows], "o-", label=kind)
    axes[1].set(title="Clean validation", xlabel="Processed training tokens", ylabel="Bits per byte (lower is better)")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
    for field in ("best_val_loss", "best_kind", "parameters", "processed_tokens", "optimizer_updates_applied", "steady_tokens_per_second", "peak_allocated_gib", "estimated_1000_full_updates_minutes"):
        print(field, ":", result[field])
    print("1,000 full effective updates contain", f"{cfg.effective_tokens * 1000:,}", "tokens.")
    print("1,000 sequences contain", f"{cfg.sequence_len * 1000:,}", "tokens at this length.")
    print("Extrapolated timings are not an official tiny-track timing result.")


In [ ]:
#@title 9. Optional generation from the selected best checkpoint
if RUN_SAMPLING and model is not None:
    best = load_tensors(Path(result["config"]["run_dir"]) / "best.pt")
    load_parameters(model, best["parameters"])
    print("Candidate:", best["metrics"]["kind"], "clean CE:", best["metrics"]["val_loss"])
    print(generate(model, encoder, "The United States of ", temperature=0.7, top_k=40))
else:
    print("Set RUN_SAMPLING=True after training. Sampling settings do not change validation loss.")


## Candidate experiment: MuonEq-R

Muon orthogonalizes a momentum matrix. **MuonEq-R first balances the norm of
each row**, reducing the dominance of unusually large rows before that operation.
It adds no prediction head or second model pass. It is already in the one-hour
trainer but **not in the pinned current tiny leader**.

The [paper](https://arxiv.org/abs/2603.28254) and
[prior tiny comparisons](https://github.com/qlabs-eng/slowrun/pull/77) make it
worth testing. They do not prove it helps on top of the latest meta-gradient
recipe. The optional cell changes only this flag across matched seed pairs.
Keep identical data hashes, exposure, checkpoint selection and validation.


In [ ]:
#@title 10. Optional matched MuonEq-R experiment (six complete training runs)
RUN_ABLATION = False #@param {type:"boolean"}
ABLATION_SEEDS = (42, 43, 44)
if RUN_ABLATION:
    model = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    comparisons = []
    for seed in ABLATION_SEEDS:
        for enabled in (False, True):
            trial_cfg = replace(cfg, seed=seed, muon_eq_r=enabled,
                                run_dir=f"runs/eqr_{stamp}_seed{seed}_{int(enabled)}")
            trial_model, trial_result, _ = train(trial_cfg, train_path, val_path, byte_counts)
            comparisons.append({"seed": seed, "muon_eq_r": enabled,
                                "best_val_loss": trial_result["best_val_loss"],
                                "wall_seconds": trial_result["wall_seconds"],
                                "processed_tokens": trial_result["processed_tokens"]})
            del trial_model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    print(json.dumps(comparisons, indent=2))
else:
    print("A/B disabled. Establish the control before spending time on six runs.")


## Official record reproduction: unmodified pinned upstream path

This section requires a host with **8 H100 GPUs of approximately 80GB each**
and the upstream **PyTorch 2.10** environment. Ordinary Colab runtimes do not
satisfy this. It is off by default and is intended for a suitable attached
runtime or an exported notebook on the benchmark host.

The cell verifies the pinned source hashes, installs upstream requirements
only after hardware/version checks, prepares the official 100M/10M data if
absent, verifies its prescribed hashes, and launches the unchanged script.
No portable-code scores are substituted for this path. It can use many GB
of disk and much longer for preprocessing/compilation/evaluation than the
leaderboard's reported training-time quantity.

The current flags are named **`--mg-*`**, not the older `--ttt-*` spelling in
the PR's initial description. In the official runner, `--mg-every 0` returns
to the prior recipe; accumulation greater than one bypasses MG, so retain
the record's 32 sequences/GPU and effective batch 524,288.


In [ ]:
#@title 11. Guarded official launcher
RUN_OFFICIAL = False #@param {type:"boolean"}
if RUN_OFFICIAL:
    if torch.cuda.device_count() != 8:
        raise RuntimeError("Official tiny reproduction requires exactly 8 H100 GPUs; use the portable path on Colab.")
    for index in range(8):
        props = torch.cuda.get_device_properties(index)
        if "H100" not in props.name.upper() or props.total_memory < 70 * 2**30:
            raise RuntimeError("Expected an 8xH100 80GB-class node.")
    if torch.__version__.split("+")[0].split(".")[:2] != ["2", "10"]:
        raise RuntimeError("Use a fresh upstream torch==2.10 environment, then restart; this cell will not replace an imported torch.")
    checkout = Path("slowrun_record_" + UPSTREAM_COMMIT[:12]).resolve()
    if not checkout.exists():
        subprocess.run(["git", "clone", "--no-checkout", "https://github.com/qlabs-eng/slowrun.git", str(checkout)], check=True)
        subprocess.run(["git", "checkout", "--detach", UPSTREAM_COMMIT], cwd=checkout, check=True)
    head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=checkout, text=True).strip()
    if head != UPSTREAM_COMMIT:
        raise RuntimeError("Existing checkout is not the pinned revision; choose a fresh directory.")
    expected_sources = {
        "tiny/train.py": "e29825be413384a6d9323e3353d8106f0bd62b9e1530e8c576a663c5c2b6165d",
        "tiny/cuda_kernels.py": "d4ce7e124280b8f3e1b2ed3b7caf7435c36a6b96c0e7b9bcdfcb3e3292a2e5f7",
        "prepare_data.py": "7bc55102af9cdce5ed97eecf66903d10e8cdb4715b0fbe46c9cdb8c0891c1b0f",
    }
    for relative, expected in expected_sources.items():
        if sha256_file(checkout / relative) != expected:
            raise RuntimeError(f"Source hash mismatch: {relative}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(checkout / "requirements.txt")], check=True)
    canonical = {
        "fineweb_train.pt": "36e7c95c1e7f6ed952fb002d76a03044e8617fea7e696a68d7dc1ce78465dcaf",
        "fineweb_val.pt": "6868ed375b289a89c72c2f9df1ecbdcff700c4b9478ca806435d2dbfad8573b1",
    }
    if not all((checkout / "fineweb_data" / name).exists() for name in canonical):
        subprocess.run([sys.executable, "prepare_data.py"], cwd=checkout, check=True)
    for name, expected in canonical.items():
        if sha256_file(checkout / "fineweb_data" / name) != expected:
            raise RuntimeError(f"Canonical data hash mismatch: {name}")
    import os
    environment = dict(os.environ)
    environment.setdefault("WANDB_MODE", "disabled")
    subprocess.run([sys.executable, "-m", "torch.distributed.run", "--standalone",
                    "--nproc_per_node=8", "tiny/train.py", "--seed", "42"],
                   cwd=checkout, env=environment, check=True)
else:
    print("Official launch disabled. Portable Colab runs are not record reproductions.")
    print("Record command: torchrun --standalone --nproc_per_node=8 tiny/train.py --seed 42")


In [ ]:
#@title 12. Download a portable run's artifacts from Colab
DOWNLOAD_RESULTS = False #@param {type:"boolean"}
if DOWNLOAD_RESULTS and result is not None:
    import shutil
    archive = shutil.make_archive(str(Path(result["config"]["run_dir"])), "zip", result["config"]["run_dir"])
    if "google.colab" in sys.modules:
        from google.colab import files
        files.download(archive)
    else:
        print("Archive:", archive)


## Reference: all methods, evidence and portability details

# Slowrun methods and the current tiny-track recipe

Verified **2026-10-19**, against upstream commit
[`c6f0fa1b415994e0c5cf163990fe253fe1622883`](https://github.com/qlabs-eng/slowrun/tree/c6f0fa1b415994e0c5cf163990fe253fe1622883).
The latest accepted tiny entry listed is **3.295 validation loss, 14.2 minutes on
one 8xH100 node**, submitted September 5 and merged September 15, 2026.
This is a verified upstream report, not a result achieved by this notebook.

Sources: [leaderboard](https://github.com/qlabs-eng/slowrun#tiny-track-15-minutes),
[record PR #99](https://github.com/qlabs-eng/slowrun/pull/99),
[pinned tiny code](https://github.com/qlabs-eng/slowrun/blob/c6f0fa1b415994e0c5cf163990fe253fe1622883/tiny/train.py).
Open proposals are not accepted records. In particular, the lower number in the
open varlen/test-time-training PR uses a different evaluation protocol.

## What counts as tiny track?

"Tiny" describes the **compute allowance**, not a 30M-parameter model. The accepted
track permits a single 8xH100 node for at most 15 minutes under the upstream timing
protocol. Training uses the prescribed 100M-token FineWeb corpus; the standard
validation file has 10M tokens. A smaller corpus, different context length, one
Colab GPU, another validation subset, or a portable numerical backend is a study
configuration, not a directly comparable leaderboard entry.

The current upstream defaults are 16 layers, width 1,024, 8 heads, context 2,048,
15 epochs, device batch 32 sequences per GPU, and an effective batch of 524,288
tokens. The recurrence schedule switches from one pass to two after 70% of the
planned updates. Backpropagation is through only the last recurrent pass.

The official launch section uses the unmodified pinned code, custom FP8 loss,
FA3 where available, distributed optimizer, original data hashes and upstream
defaults. The portable section provides the current *algorithmic recipe* for
experiments. It does not promise identical numerics, speed or validation loss.

## Every tiny-track record milestone

These are cumulative leaderboard entries, not isolated ablations. A difference
between two rows is not proof that the named change alone caused the whole gain.

| Entry | Reported val loss | Change and high-level explanation |
|---|---:|---|
| 1 | 3.428 | Approximately 300M-parameter transformer, Muon, dropout 0.1 and weight decay 0.8. Reuse the limited corpus for multiple epochs; regularize so a large model can learn without simply memorizing. |
| 2 | 3.410 | **SwiGLU:** a learned SiLU gate multiplies a second MLP projection, giving each token a way to select which features pass through. |
| 3 | 3.395 | **U-Net connections:** later blocks receive outputs saved from earlier blocks. These are extra paths through depth, beyond ordinary residual connections. |
| 4 | 3.385 | **Per-head attention gates:** a context-dependent sigmoid scales each attention head's output, allowing that head to contribute less when it is unhelpful. |
| 5 | 3.383 | **Warmdown tuning:** adjust when the learning rate starts falling, balancing exploration early and settling late. Current warmdown ratio is 0.4, with a separate SWA cycle. |
| 6 | 3.374 | **Half-truncated RoPE, partial key offset, residual scaling:** rotate only half the head dimensions, shift the stationary part of long-window keys by one position, and initialize trainable residual coefficients at 1.1. These change position/content interactions and signal flow. |
| 7 | 3.365 | **Weight-decay schedule:** hold strong decay initially, lower it in the middle, then raise it late. Current settings are 0.8 through epoch 2, decrease to 0.1 at epoch 8, and rise toward 0.93. |
| 8 | 3.353 | **EMA:** maintain an exponential moving average of parameters. The average is less sensitive to a noisy final update, and is evaluated as a separate candidate. |
| 9 | 3.345 | **SWA plus checkpoint averaging:** cycle the late learning rate to collect different nearby checkpoints, then average their weights. The current final four epoch snapshots receive increasing recency weights. This produces one model at inference. |
| 10 | 3.332 | **Document reshuffling:** reshuffle whole documents, concatenate them again, and repack each epoch. Merely shuffling fixed batches would keep the same context boundaries forever. |
| 11 | 3.329 | **Learnable XSA:** subtract a learnable amount of the attention output component aligned with the current token's value vector. It reduces an easy self-copy direction; it is not simply a mask on the attention diagonal. The initial entry used early layers. |
| 12 | 3.324 | **Two-pass recurrence:** run the same network again on its own latent state, reusing parameters. Extra computation can refine representations without adding another full set of layers. |
| 13 | 3.315 | **FP8 shared-head MTP and all-layer XSA:** score one output distribution against several future-token offsets; anneal the extra offsets away. A fused FP8 kernel makes the large vocabulary projection and loss affordable. Expand XSA to all layers. |
| 14 | 3.307 | **Combine recurrence with the newer recipe:** integrate two-pass recurrence with the fused MTP/XSA changes rather than treating those branches as separate alternatives. |
| 15 | 3.295 | **First-order meta-gradient:** split each rank's batch. Take the first half's gradient, temporarily move selected MLP matrices a fixed distance downhill, take the second half's gradient there, restore the weights, and give the averaged gradients to Muon. This trains for useful behavior after a small adaptation step. |

The record's meta-gradient uses `mg_every=1`, `mg_step_norm=0.5`, and the MLP
matrices. It is **training-time** adaptation using training examples, not training
on validation labels. Only the official single-microbatch distributed path is
the exact record implementation; this notebook extends the episode to a
single-device effective batch assembled with gradient accumulation.

## Baseline mechanisms that are easy to overlook

The code contains more than the named leaderboard additions:

- **Muon for matrix weights, AdamW for other groups:** Muon shapes momentum
  matrices with a five-step Polar Express/Newton-Schulz orthogonalization, then
  applies variance scaling. Embeddings, output weights, gates and small learned
  coefficients use separately tuned AdamW groups. These groups have different
  learning rates; many scalar groups intentionally have no weight decay.
- **Cautious Muon weight decay:** decay is applied where the update and weight
  signs agree. This is not identical to applying ordinary AdamW decay everywhere.
- **Value projections from the initial embedding:** selected layers project the
  original normalized token representation into an additional value signal.
  Learned gates control how much is added to the attention values. This reuses
  the input embedding instead of learning independent vocabulary-sized value tables.
- **RMS normalization and normalized Q/K:** control activation scale, including
  the vectors used in attention.
- **Alternating local/full attention:** the `SSSL` pattern uses three shorter
  context windows then a long one; the final layer always uses full context.
  This controls compute while retaining routes for longer-range information.
- **Original-input injection:** each layer mixes its recurrent state with the
  initial token embedding through learned `resid_lambdas` and `x0_lambdas`.
  The original embedding remains the anchor on every recurrent pass.
- **Initialization:** embeddings have standard deviation 1; the output head
  starts small; attention and MLP output projections start at zero. These
  choices shape the initial residual behavior. The tiny leader still uses zero
  output projections, despite a different initialization change on the one-hour track.
- **Logit softcap:** apply `15*tanh(logits/15)` before cross-entropy to limit
  extreme logits. This is separate from changing sampling temperature.
- **Gradient accumulation:** sequence microbatches contribute to one larger
  optimizer update. It changes memory use and update frequency; it is not
  additional unique training data.
- **Implementation speed:** compilation, BF16, FA3, fused FP8 projection/loss,
  and distributed optimizer sharding reduce runtime or memory. They do not
  individually guarantee better predictions; in a timed track they can leave
  room for a more expensive learning algorithm.

The portable implementation uses SDPA, FP32 master weights/states and ordinary
autograd. It does not emulate FP8 quantization. On BF16-capable CUDA GPUs it uses
BF16 autocast and Muon orthogonalization. On T4 it defaults to FP32 because T4
does not provide native BF16. Explicit FP16 uses gradient scaling; its numerical
behavior needs checking on the actual GPU.

## MTP: an important correction to the earlier notebook plan

Current tiny MTP has **no extra vocabulary heads**. A single set of logits is
scored against flattened targets at offsets 0, 1 and 2, initially weighted
1.0, 0.5 and 0.25. The further offset vanishes first; all extra offsets are zero
after 66% of the schedule. Its training loss is a weighted sum, so it should not
be plotted as though it were the same quantity as clean next-token validation CE.

The portable implementation matches the kernel's flattened-offset convention,
including offset targets that cross a packed sequence row boundary. Its training
softmax also includes the padded vocabulary, while standard evaluation slices
to the tokenizer's real vocabulary, matching upstream's distinction. These are
reasons to test the port carefully rather than claim numerical equivalence.

The earlier idea of adding another full 50k-way head is a different experiment.
It would add parameters; it is not necessary to follow the current tiny recipe.

## Other methods implemented elsewhere in Slowrun

The following catalog covers the other distinct accepted leaderboard approaches.
They should not all be turned on in tiny merely because they helped another track.
Source: the corresponding pinned
[one-hour trainer](https://github.com/qlabs-eng/slowrun/blob/c6f0fa1b415994e0c5cf163990fe253fe1622883/train.py),
[two-hour trainer](https://github.com/qlabs-eng/slowrun/blob/c6f0fa1b415994e0c5cf163990fe253fe1622883/two_hour/train.py), and
[unlimited trainer](https://github.com/qlabs-eng/slowrun/blob/c6f0fa1b415994e0c5cf163990fe253fe1622883/unlimited/train.py).

| Method | Explanation | Tiny status at the pinned revision |
|---|---|---|
| Larger models with heavier regularization | More capacity can help limited-data learning when decay/dropout control overfitting. More parameters alone can worsen validation. | Smaller approximately 300M recipe, with hardware-scaled study presets. |
| Repeating selected middle layers | Reuse a subset of blocks later in training, increasing effective depth without new parameters. | Tiny instead uses scheduled whole-network recurrence. |
| Interleaved Head Attention (IHA) | Learn small mixing matrices across attention heads; identity initialization preserves the starting baseline. | Not in the current tiny leader. |
| MuonEq-R | Normalize rows of the momentum/update matrix before orthogonalization so dominant rows do not dictate its geometry. | In the one-hour trainer; optional experiment added to this study port. |
| Normal rather than zero output-projection initialization | Let residual branches contribute immediately instead of starting at zero; interacts with the rest of the recipe. | One-hour historical improvement, not the current tiny initialization. |
| Stochastic depth | Randomly skip residual blocks during training, regularizing depth and sometimes saving work. | One-hour historical improvement; not enabled in current tiny. |
| Weighted checkpoint combinations | Favor recent parameter snapshots over older ones. | Current tiny averages the retained late checkpoints with weights 1,2,3,4. |
| Context-window scheduling | Adjust attended context during training, trading short-window efficiency for long-window learning. | Described on the two-hour history; tiny uses its fixed `SSSL` windows. |
| Longer training and model/ensemble scaling | Spend more compute replaying the same prescribed data and/or training more models. | Must fit tiny's time budget to be relevant. |
| Logit ensembles | Average several models' pre-softmax scores before scoring. | Unlimited track; additional inference models are expensive. |
| Probability ensembles | Average predicted probabilities instead, allowing a mixture of models rather than a single consensus logit vector. | Unlimited track; distinct from parameter averaging. |
| Sequential self-distillation | Train a new model against previous models' predictions as well as real next-token labels, reusing the same corpus. | Unlimited track; adds teacher training/inference cost. |
| Snapshot ensembles | Retain several checkpoints of each training trajectory as ensemble members. | Unlimited track; increases prediction and storage cost. |
| Gradient-based ensemble selection | Use a validation-related objective to choose/combine useful candidate models. | Unlimited track; follow upstream selection/evaluation rules rather than tuning on a final test set. |

The research directory additionally contains **universal/recurrent transformer**
experiments and **hybrid full/linear attention** with GatedDeltaNet and Kimi Delta
Attention backends. The latter use a recurrent state instead of a full quadratic
attention matrix in selected layers. They are exploratory research recipes,
not extra accepted tiny-record gains. The open PR for variable-length
document attention, strided evaluation and test-time training changes context or
evaluation semantics; its scores must not be compared directly with standard
tiny validation loss.

## Recommended next experiment: MuonEq-R

Imagine one gradient matrix has a few rows with much larger magnitudes than the
rest. Ordinary Muon orthogonalizes that unbalanced matrix. MuonEq-R first divides
each row by its norm, then runs the same orthogonalization. The proposal is that
balancing the rows gives this finite-step transformation a more useful geometry.
It does not add a model head, a second forward pass, or extra training examples.

Why it is a reasonable experiment here:

1. The current tiny script already uses Muon, so the code change is localized.
2. The [MuonEq paper](https://arxiv.org/abs/2603.28254) reports improvements from
   pre-orthogonalization balancing; this is evidence for trying it, not a guarantee.
3. [PR #77](https://github.com/qlabs-eng/slowrun/pull/77) reported two matched tiny
   pairs with approximately 0.0013 and 0.0028 lower checkpoint-averaged loss.
   That PR was closed rather than merged in that form. The current tiny leader's
   optimizer still lacks row normalization, although it is in the one-hour code.
4. Its interaction with the newer meta-gradient and recurrence recipe is unproven.

Use `muon_eq_r=False` as the record-inspired control, then change only that flag
to `True`. Compare the same seeds, data hashes, exposure, validation tokens and
checkpoint policy. Record clean CE/BPB, allocated/reserved VRAM, wall time and
processed tokens/sec. Repeat a promising result on at least three seeds, then
test on the official runner and hardware before making a record claim.

## Portability changes and evaluation discipline

- Both official and study data use GPT-2 BPE. Study preprocessing consumes the
  full 10M-token official validation prefix before taking any training documents,
  even if the study retains only a small validation subset. Changing validation
  size must not shift the training source into the canonical validation corpus.
- Study caches include tokenizer/source/budget metadata and verify saved hashes.
  Official reproduction uses upstream's original files and prescribed hashes.
- The study loop processes a smaller final accumulation group rather than
  accidentally training beyond the requested epoch. It records actual tokens.
  This is a deliberate scheduling difference from the exact upstream runner.
- All study validation candidates use the same maximum recurrence count and
  clean next-token CE/BPB. Do not compare the extra MTP training objective to CE.
- Model, EMA and recency-weight-averaged validation results are reported
  separately. `best.pt` records which candidate won. Generation loads that
  candidate rather than accidentally using a worse final live checkpoint.
- The probe measures early and late recurrence shapes with MG, MTP and EMA
  active. Leave memory headroom; the largest tested fit is not the true maximum.
- Study time includes measured training and separately records end-to-end loop
  time. Neither is a claim to have met the upstream 15-minute timing protocol.

## Suggested workflow

1. Run the included synthetic CPU checks; no FineWeb or GPU download is needed.
2. Upload the notebook to Colab, choose GPU, and use `smoke` first.
3. Select a hardware study preset, inspect the parameter summary, and run the probe.
4. Reproduce the current recipe on a fixed study corpus; save config and results.
5. Run the optional MuonEq-R A/B cell. Keep every other setting fixed.
6. Move a promising change into the pinned upstream script on 8xH100 with the
   official data and evaluation protocol. Full records require that final test.
